# 5f — Master Unified S2D Diagnostic Architecture
## Product Integration, Model-Attractor Interpretation, and IC–Drift Attribution

This production notebook consumes standardized products from 5b–5e. It does not recalculate producer diagnostics.

0. **Branch 0 — IC fingerprints:** native-grid candidate drivers from 5b.
1. **Branch A — Field drift:** standardized regional, monthly, and daily responses from 5a/5c/5d.
2. **Branch B — Physical pathways:** coupling relationships and budget consistency from 5e.
3. **Branch C — Model attractor:** movement toward observations and an independent uninitialized E3SM historical climatology matched by calendar date.
4. **Branch D — IC-to-drift attribution:** weighted spatial alignment, projection, and across-start regression.

Paired values use `JRA55_FOSIRL − Reanalysis`; initialization years are the paired bootstrap blocks. Causal language remains cautious until spatial alignment, cross-start association, and a consistent physical pathway agree.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import sys
import esp_lab

# Development-checkout fallback: a clean install exposes ``workflows`` directly.
REPO_ROOT = Path(esp_lab.__file__).resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from workflows.diagnostics.unified import branch_a_drift as field_drift
from workflows.diagnostics.unified import branch_b_physics as physical_consistency
from workflows.diagnostics.unified import branch_c_attractor as model_attractor
from workflows.diagnostics.unified import config as configuration
from workflows.diagnostics.unified import inventory as inventory_workflow
from workflows.diagnostics.unified import references as reference_workflow
from workflows.diagnostics.unified import run_unified as unified_workflow

from esp_lab.diagnostics.attractor_core import (
    MovementCategory, spatial_rmsd,
    compute_attractor_distances, compute_relative_movement,
    classify_movement_category, classify_spatial_movement,
    build_2axis_trajectory,
)
from esp_lab.utils.dask_utils import DaskConfig, get_cluster_client, close_cluster

UNIFIED_DIR = Path(configuration.__file__).resolve().parent
print('Unified S2D Pipeline imports OK')


## Dask Setup

In [ ]:
machine_env = os.environ.get('CLUSTER_TYPE', 'local')
dask_cfg = DaskConfig(cluster_type=machine_env, workers=8, cores=4, memory='16GB')
cluster, client = get_cluster_client(dask_cfg)
print(client)


## Step 1: Multi-Source Inventory & Readiness Gate

In [ ]:
%%time
S2D_DIAG_ROOT = Path(os.environ.get('ESP_LAB_S2D_DIAG_ROOT', '/global/cfs/cdirs/e3sm/S2S2D/s2d_diag'))
FIGURE_ROOT = Path(os.environ.get('ESP_LAB_FIGURE_OUTDIR', '/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag'))
OUTPUT_ROOT = S2D_DIAG_ROOT / 'multimodel' / 'unified_diagnostics'
FIGURE_OUTDIR = FIGURE_ROOT / 'unified_diagnostics'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_OUTDIR.mkdir(parents=True, exist_ok=True)
inv_summary = inventory_workflow.run_inventory(output_root=OUTPUT_ROOT, verbose=True)


## Branch A: Field Drift ($B^{\text{obs}}$, $D^{\text{obs}}$, $J$)

In [ ]:
%%time
# 5f consumes standardized 5b-5e products; it does not recompute them.
USE_SYNTHETIC_DEMO = False
PRODUCT_BUNDLES = sorted(S2D_DIAG_ROOT.rglob("product_manifest.json"))
if not PRODUCT_BUNDLES:
    raise FileNotFoundError("Run 5b-5e producers first; no product manifests were found.")

unified_results = unified_workflow.run(
    output_root=OUTPUT_ROOT, product_bundles=PRODUCT_BUNDLES, verbose=True,
)
product_table = unified_results["products"]["table"]
display(product_table.groupby(["workflow", "metric_name"], dropna=False).size())


## Branch B: Physical Consistency ($EF$, $BR$, Coupling)

In [ ]:
if USE_SYNTHETIC_DEMO:
    raise NotImplementedError("Synthetic Branch B is disabled in the production notebook.")


## Branch C: Model Attractor (Movement Toward Model State)

Calculates:
- Distance to Obs $d^{\text{obs}}(\tau) = \| X(\tau) - O_X \|$
- Distance to E3SM Climatology $d^{\text{model}}(\tau) = \| X(\tau) - M_X \|$
- Drift changes $\delta d^{\text{obs}}(W) = d^{\text{obs}}(W) - d^{\text{obs}}(W_0)$ and $\delta d^{\text{model}}(W) = d^{\text{model}}(W) - d^{\text{model}}(W_0)$
- Initial displacements $d^{\text{obs}}(W_0)$ and $d^{\text{model}}(W_0)$
- 4-Category Movement Classification Map
- 2-Axis Regional Trajectory Curve ($x = \delta d^{\text{model}}, y = \delta d^{\text{obs}}$)


In [ ]:
if USE_SYNTHETIC_DEMO:
    raise NotImplementedError("Synthetic Branch C is disabled; use an independent E3SM historical climatology.")


### 2-Axis Regional Trajectory Plot ($x = \delta d^{\text{model}}, y = \delta d^{\text{obs}}$)

In [ ]:
# Branch D products (r_ic, beta_ic, and across-start regression) are added by
# unified_workflow.run when IC and paired-drift native-grid fields are supplied.
product_table[product_table["metric_name"].astype(str).str.contains("attribution|r_ic|beta_ic", case=False, regex=True)]


## Shutdown

In [ ]:
close_cluster(cluster, client)
